# Docker for AI Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Install Docker

In [ ]:
```bash

# macOS

brew install --cask docker

open /Applications/Docker.app

# Ubuntu

curl -fsSL https://get.docker.com | sh

sudo usermod -aG docker $USER

# Log out and back in for group change to take effect

In [ ]:
```

Verify:

In [ ]:
```bash

docker --version

docker run hello-world

In [ ]:
```

### Step 2: Install NVIDIA Container Toolkit (Linux with NVIDIA GPU)

This lets Docker containers access your GPU. macOS and Windows (WSL2) users can skip this; Docker Desktop handles GPU passthrough differently on those platforms.

In [ ]:
```bash

distribution=$(. /etc/os-release;echo $ID$VERSION_ID)

curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg

curl -s -L https://nvidia.github.io/libnvidia-container/$distribution/libnvidia-container.list | \

    sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \

    sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list

sudo apt-get update

sudo apt-get install -y nvidia-container-toolkit

sudo nvidia-ctk runtime configure --runtime=docker

sudo systemctl restart docker

In [ ]:
```

Test GPU access inside a container:

In [ ]:
```bash

docker run --rm --gpus all nvidia/cuda:12.4.1-base-ubuntu22.04 nvidia-smi

In [ ]:
```

If you see your GPU info, the toolkit is working.

### Step 3: Understand base images

Choosing the right base image saves hours of debugging.

In [ ]:
```

nvidia/cuda:12.4.1-devel-ubuntu22.04

  Full CUDA toolkit. Compilers included.

  Use for: building packages that need nvcc (flash-attn, bitsandbytes)

  Size: ~4 GB

nvidia/cuda:12.4.1-runtime-ubuntu22.04

  CUDA runtime only. No compilers.

  Use for: running pre-built code

  Size: ~1.5 GB

pytorch/pytorch:2.3.1-cuda12.4-cudnn9-runtime

  PyTorch pre-installed on top of CUDA.

  Use for: skipping the PyTorch install step

  Size: ~6 GB

python:3.12-slim

  No CUDA. CPU only.

  Use for: inference on CPU, lightweight tools

  Size: ~150 MB

In [ ]:
```

### Step 4: Write a Dockerfile for AI development

Here is the Dockerfile in `code/Dockerfile`. Walk through it:

In [ ]:
```dockerfile

FROM nvidia/cuda:12.4.1-devel-ubuntu22.04

ENV DEBIAN_FRONTEND=noninteractive

ENV PYTHONUNBUFFERED=1

RUN apt-get update && apt-get install -y --no-install-recommends \

    python3.12 \

    python3.12-venv \

    python3.12-dev \

    python3-pip \

    git \

    curl \

    build-essential \

    && rm -rf /var/lib/apt/lists/*

RUN update-alternatives --install /usr/bin/python python /usr/bin/python3.12 1

RUN python -m pip install --no-cache-dir --upgrade pip setuptools wheel

RUN python -m pip install --no-cache-dir \

    torch==2.3.1 \

    torchvision==0.18.1 \

    torchaudio==2.3.1 \

    --index-url https://download.pytorch.org/whl/cu124

RUN python -m pip install --no-cache-dir \

    numpy \

    pandas \

    scikit-learn \

    matplotlib \

    jupyter \

    transformers \

    datasets \

    accelerate \

    safetensors

WORKDIR /workspace

VOLUME ["/workspace", "/models"]

EXPOSE 8888

CMD ["python"]

In [ ]:
```

Build it:

In [ ]:
```bash

docker build -t ai-dev -f phases/00-setup-and-tooling/07-docker-for-ai/code/Dockerfile .

In [ ]:
```

This takes a while the first time (downloading CUDA base image + PyTorch). Subsequent builds use cached layers.

Run it:

In [ ]:
```bash

docker run --rm -it --gpus all \

    -v $(pwd):/workspace \

    -v ~/models:/models \

    ai-dev python -c "import torch; print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')"

In [ ]:
```

Run Jupyter inside the container:

In [ ]:
```bash

docker run --rm -it --gpus all \

    -v $(pwd):/workspace \

    -v ~/models:/models \

    -p 8888:8888 \

    ai-dev jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root

In [ ]:
```

### Step 5: Volume mounts for data and models

Volume mounts are critical for AI work. Without them, your 14 GB model downloads vanish when the container stops.

In [ ]:
```bash

# Mount your code

-v $(pwd):/workspace

# Mount a shared models directory

-v ~/models:/models

# Mount datasets

-v ~/datasets:/data

In [ ]:
```

Inside your training script, load from the mounted path:

In [ ]:
```python

from transformers import AutoModel

model = AutoModel.from_pretrained("/models/llama-7b")

In [ ]:
```

The model lives on your host filesystem. Rebuild the container as often as you want without re-downloading.

### Step 6: Docker Compose for multi-service AI apps

A real RAG application needs an inference server and a vector database. Docker Compose runs both with one command.

See `code/docker-compose.yml`:

In [ ]:
```yaml

services:

  ai-dev:

    build:

      context: .

      dockerfile: Dockerfile

    deploy:

      resources:

        reservations:

          devices:

            - driver: nvidia

              count: all

              capabilities: [gpu]

    volumes:

      - ../../../:/workspace

      - ~/models:/models

      - ~/datasets:/data

    ports:

      - "8888:8888"

    stdin_open: true

    tty: true

    command: jupyter notebook --ip=0.0.0.0 --port=8888 --no-browser --allow-root

  qdrant:

    image: qdrant/qdrant:v1.12.5

    ports:

      - "6333:6333"

      - "6334:6334"

    volumes:

      - qdrant_data:/qdrant/storage

volumes:

  qdrant_data:

In [ ]:
```

Start everything:

In [ ]:
```bash

cd phases/00-setup-and-tooling/07-docker-for-ai/code

docker compose up -d

In [ ]:
```

Now your AI dev container can reach the vector database at `http://qdrant:6333` by service name. Docker Compose creates a shared network automatically.

Test the connection from inside the AI container:

In [ ]:
```python

from qdrant_client import QdrantClient

client = QdrantClient(host="qdrant", port=6333)

print(client.get_collections())

In [ ]:
```

Stop everything:

In [ ]:
```bash

docker compose down

In [ ]:
```

Add `-v` to also delete the qdrant volume:

In [ ]:
```bash

docker compose down -v

In [ ]:
```

### Step 7: Useful Docker commands for AI work

In [ ]:
```bash

# List running containers

docker ps

# List all images and their sizes

docker images

# Remove unused images (reclaim disk space)

docker system prune -a

# Check GPU usage inside a running container

docker exec -it <container_id> nvidia-smi

# Copy a file from container to host

docker cp <container_id>:/workspace/results.csv ./results.csv

# View container logs

docker logs -f <container_id>

In [ ]:
```

## Exercises

In [ ]:
1. Build the Dockerfile and run `python -c "import torch; print(torch.__version__)"` inside the container
2. Start the docker-compose stack and verify Qdrant is accessible from the AI container at `http://qdrant:6333/collections`
3. Add `flask` to the Dockerfile, rebuild, and run a simple API server on port 5000. Map the port with `-p 5000:5000`
4. Measure the image size with `docker images`. Try switching the base image from `devel` to `runtime` and compare sizes